# 独立测试集负荷预测与评价

本 Notebook 加载已经训练好的 Transformer 检查点，对独立测试集按 24 小时非重叠窗口进行预测，并输出总体指标、逐预测步长指标和真实/预测负荷曲线。

> 当前默认数据是项目 A 训练集，仅用于检查流程。使用训练数据得到的指标不能代表模型泛化性能，正式评价时请在下方配置单元替换为独立测试集。

## 1. 配置路径与运行参数

只需要修改 `DATA_PATH` 和 `CHECKPOINT_PATH`。支持 `LoadTransformer`、`LoadTransformerHistoryOnly` 和 `LoadTransformerEncoderDecoder`。

In [ ]:
from pathlib import Path

DATA_PATH = Path("附件3：训练数据集/训练数据项目A历史数据_2025-04-01_2025-10-31.xlsx")
CHECKPOINT_PATH = Path("outputs/LoadTransformer_weather.pt")
OUTPUT_DIR = Path("outputs/test_evaluation")
DEVICE = "auto"  # 可选：auto、cpu、cuda
BATCH_SIZE = 64

print("测试数据：", DATA_PATH.resolve())
print("模型检查点：", CHECKPOINT_PATH.resolve())
print("输出目录：", OUTPUT_DIR.resolve())

## 2. 导入评价函数

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from evaluate_test import (
    build_prediction_frame,
    build_test_windows,
    calculate_metrics,
    calculate_metrics_by_horizon,
    predict_test_windows,
    save_evaluation_outputs,
)
from load_forecasting.checkpoint import load_checkpoint
from load_forecasting.data import load_excel
from train import select_device

## 3. 加载检查点和测试数据

测试集会使用与训练时一致的清洗、小时聚合、窗口内降噪和小波分解流程。

In [ ]:
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"找不到模型检查点：{CHECKPOINT_PATH}")
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"找不到测试数据：{DATA_PATH}")

checkpoint = load_checkpoint(CHECKPOINT_PATH, map_location="cpu")
test_frame = load_excel(DATA_PATH)

print("模型类型：", checkpoint["model_type"])
print("小时数据形状：", test_frame.shape)
print("数据时间范围：", test_frame["timeStamp"].iloc[0], "至", test_frame["timeStamp"].iloc[-1])

## 4. 构造非重叠测试窗口

最前面的 168 小时只作为历史上下文。此后窗口每次移动 24 小时，因此同一个真实时刻不会被重复预测。

In [ ]:
windows = build_test_windows(test_frame, checkpoint)

print("历史输入：", windows["history"].shape)
print("未来天气：", windows["future_weather"].shape)
print("真实负荷：", windows["target"].shape)
print("候选窗口：", windows["candidate_windows"])
print("成功窗口：", windows["successful_windows"])
print("跳过窗口：", windows["skipped_windows"] )

## 5. 加载模型并执行批量预测

In [ ]:
device = select_device(DEVICE)
predicted = predict_test_windows(
    checkpoint,
    windows["history"],
    windows["future_weather"],
    device,
    batch_size=BATCH_SIZE,
)
actual = windows["target"]

print("预测设备：", device)
print("预测结果：", predicted.shape)

## 6. 总体性能指标

MAE 和 RMSE 使用全部有效时段；MAPE 只使用真实负荷非零的时段。

In [ ]:
overall_metrics = calculate_metrics(predicted, actual)
pd.DataFrame([overall_metrics])

## 7. h+1 至 h+24 分步指标

该表用于观察预测误差是否随预测步长增加而增大。

In [ ]:
metrics_by_horizon = calculate_metrics_by_horizon(predicted, actual)
metrics_by_horizon

## 8. 查看逐小时预测明细

In [ ]:
prediction_detail = build_prediction_frame(
    windows["timestamps"], actual, predicted
)
prediction_detail.head(24)

## 9. 绘制真实负荷与预测负荷曲线

In [ ]:
figure, axis = plt.subplots(figsize=(16, 6))
axis.plot(
    prediction_detail["timeStamp"],
    prediction_detail["actual_load"],
    label="Actual load",
)
axis.plot(
    prediction_detail["timeStamp"],
    prediction_detail["predicted_load"],
    label="Predicted load",
)
axis.set_title(
    f"Test load forecast | MAE={overall_metrics['mae']:.3f}, "
    f"RMSE={overall_metrics['rmse']:.3f}, MAPE={overall_metrics['mape']:.3f}%"
)
axis.set_xlabel("Time")
axis.set_ylabel("Load")
axis.grid(alpha=0.25)
axis.legend()
figure.autofmt_xdate()
figure.tight_layout()
plt.show()

## 10. 保存评价结果

生成 `predictions.csv`、`metrics.json`、`metrics_by_horizon.csv` 和 `load_curve.png`。

In [ ]:
save_evaluation_outputs(
    prediction_detail,
    overall_metrics,
    metrics_by_horizon,
    OUTPUT_DIR,
)

print("评价结果已保存：")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path)